In [89]:
import numpy as np
import pandas as pd
from tqdm import tqdm

%load_ext autoreload
%autoreload 2
from plotting import *

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


# Economic Dispatch & Unit Commitment in a Simple Power Market

## Background

You are an energy modeller at a consultancy. A grid operator needs to schedule a fleet 
of generators over a 24-hour period to meet hourly electricity demand at minimum cost.

This is one of the most fundamental problems in power systems: the **Unit Commitment 
and Economic Dispatch** problem. It combines two decisions:

- **Unit Commitment (UC)**: *Which* generators should be switched on each hour? 
  (binary on/off decision)
- **Economic Dispatch (ED)**: Given which generators are on, *how much* should each 
  produce? (continuous allocation decision)

---

## The Generator Fleet

You have five generators with the following characteristics:

| Generator | Type       | Capacity (MW) | Marginal Cost (£/MWh) | Startup Cost (£) | Min Run Time (hrs) | Min Stable Generation (MW) |
|-----------|------------|---------------|------------------------|------------------|--------------------|--------------------|
| G1        | Nuclear    | 400           | 10                     | 50,000           | 8                  | 150                |
| G2        | Coal       | 300           | 25                     | 8,000            | 4                  | 100                |
| G3        | Gas (CCGT) | 200           | 45                     | 2,000            | 2                  | 60                 |
| G4        | Gas (OCGT) | 100           | 80                     | 500              | 1                  | 0                  |
| G5        | Oil Peaker | 50            | 120                    | 200              | 1                  | 0                  |

**Notes:**
- All generators are offline at t=0 (start of the day)
- Capacity = maximum amount of power a generator can produce
- Marginal cost = cost per MWh of electricity produced while running
- Startup cost = one-time cost incurred each time a generator is switched on
- Min run time = once switched on, a generator cannot be switched off before this many hours
- Min Stable Generation = minimum amount of power a generation that is on must produce in order to be stable

---

## Demand Profile

Hourly demand (MW) over 24 hours:

```python
demand = [
    280, 265, 250, 245, 245, 260,   # 00:00 - 05:00 (night trough)
    310, 420, 480, 510, 530, 545,   # 06:00 - 11:00 (morning ramp)
    540, 530, 380, 375, 510, 555,   # 12:00 - 17:00 (midday plateau)
    610, 625, 590, 545, 460, 365,   # 18:00 - 23:00 (evening peak then falloff)
]
```

### Part 1 — Merit Order Dispatch (Greedy Baseline)

Implement a **merit order dispatcher** that, for each hour:
1. Ignores startup costs and minimum run times entirely
2. Ranks generators by marginal cost (cheapest first)
3. Dispatches generators in that order until demand is met
4. Sets the **market clearing price** as the marginal cost of the last (most expensive) 
   generator needed to meet demand — this is the **System Marginal Price (SMP)**

**Output:**
- For each hour: which generators run, how much each produces, and the SMP
- Total production cost across the day (marginal costs only, no startup costs)
- A stacked bar chart of hourly generation mix by generator type
- A line plot of the System Marginal Price over 24 hours

**Question to answer:** What is the total generation cost under the greedy merit order?

In [90]:
class Generator:
    """A class to represent a power generator with its characteristics."""
    def __init__(self, name, gen_type, capacity, mcost, scost, min_runtime, min_output):
        self.name = name
        self.type = gen_type                # e.g. 'nuclear', 'coal', 'gas', 'wind', 'solar'
        self.capacity = capacity            # in MW
        self.mcost = mcost                  # marginal cost in £/MWh
        self.scost = scost                  # start-up cost in £
        self.min_runtime = min_runtime      # minimum runtime in hours once started
        self.min_output = min_output        # minimum output in MW when the generator is on

        assert self.capacity >= self.min_output, f"Generator {self.name} has min_output greater than capacity, which is not feasible."
        
    def cost(self, output):
        """Calculate the total cost for a given output (in MW)."""
        if output > self.capacity:
            raise ValueError(f"Output {output} exceeds generator capacity of {self.capacity} MW.")
        return self.mcost * output
    
    def __repr__(self):
        return f"Generator(name={self.name}, type={self.type}, capacity={self.capacity} MW, mcost={self.mcost} £/MWh, scost={self.scost} £, min_runtime={self.min_runtime} h)"


In [91]:
# Given data: a list of generators with their characteristics
generators = [
    Generator('g1', 'nuclear', 400, 10, 50e3, 8, 150),
    Generator('g2', 'coal', 300, 25, 8e3, 4, 100),
    Generator('g3', 'gas-ccgt', 200, 45, 2e3, 2, 60),
    Generator('g4', 'gas-ocgt', 100, 80, 500, 1, 0),
    Generator('g5', 'oil', 50, 120, 200, 1, 0),
]

# No need to sort by marginal cost because it is already in ascending order,
# but otherwise we would sort like this:
# Sort generators by marginal cost (ascending)
# generators.sort(key=lambda g: g.mcost)

In [92]:
# Given data: hourly demand for 24 hours (in MW)
demand = [
    280, 265, 250, 245, 245, 260,   # 00:00 - 05:00 (night trough)
    310, 420, 480, 510, 530, 545,   # 06:00 - 11:00 (morning ramp)
    540, 530, 380, 375, 510, 555,   # 12:00 - 17:00 (midday plateau)
    610, 625, 590, 545, 460, 365,   # 18:00 - 23:00 (evening peak then falloff)
]

In [93]:
plot_hourly(demand, title="Hourly Demand (MW)", yaxis_title="Demand (MW)", color='purple')

In [94]:
# Stores the generators used and their outputs for each hour, and the corresponding SMPs
# `generators_by_hour` is a list of lists, where each inner list contains tuples of (generator, output) for that hour.
# For example, generators_by_hour[0] might contain [(g1, 280)], meaning that in the first hour, generator g1 is used to produce 280 MW.
# `start_up_by_hour` is a list of lists, where each inner list contains the generators that were started up in that hour.
# `smps` is a list of the SMP for each hour, which is the marginal cost of the last generator used to meet the demand.
generators_by_hour = []
start_up_by_hour = []
smps = []

generators_on = set()  # Keep track of which generators are currently on to calculate start-up costs

# Loop through each hour's demand
for dmd in tqdm(demand, desc="Calculating generation mix and SMP for each hour"):
    gens_in_hour = []
    start_ups_in_hour = []

    # Check each generator in order of increasing marginal cost.
    # For each generator, determine how much it can contribute to meeting the demand. 
    # We keep adding generators until the demand is met or we run out of generators.
    for gen in generators:
        if dmd <= 0:
            break
        output = min(gen.capacity, dmd)
        if output > 0:
            output = max(output, gen.min_output)  # Ensure we meet the minimum output if the generator is on
            gens_in_hour.append((gen, output))
        dmd -= output
    
    # Raise a warning if demand is not met after using all generators
    if dmd > 0:
        print(f"Warning: Demand not met for hour with remaining unmet demand of {dmd} MW.")

    # Append the generators in this our and the marginal cost of the last generator
    generators_by_hour.append(gens_in_hour)
    smps.append(max(gen.mcost for gen, _ in gens_in_hour))

    # Check which, if any, generators were started up in this hour
    for gen, _ in gens_in_hour:
        if gen not in generators_on:
            start_ups_in_hour.append(gen)
    generators_on = set(gen for gen, _ in gens_in_hour)  # Update the set of generators that are on
    start_up_by_hour.append(start_ups_in_hour)


Calculating generation mix and SMP for each hour: 100%|██████████| 24/24 [00:00<00:00, 285975.27it/s]


In [95]:
plot_stacked_generators(generators_by_hour, title="Hourly Generation by Generator Type (MW)", yaxis_title="Generation (MW)")

In [96]:
plot_hourly([len(start_ups) for start_ups in start_up_by_hour], title="Number of Start-Ups per Hour - Greedy Solution", yaxis_title="Number of Start-Ups", color='red', line=False)

# print which generators were started up in each hour
for t, start_ups in enumerate(start_up_by_hour):
    if start_ups:  # Only print hours where there were start-ups
        gen_names = ", ".join(gen.name + f"({gen.type})" for gen in start_ups)
        print(f"Start ups at Hour {t}: {gen_names}")

Start ups at Hour 0: g1(nuclear)
Start ups at Hour 7: g2(coal)
Start ups at Hour 16: g2(coal)


In [97]:
# Plot of the SMPs for each hour
plot_hourly(smps, title="Hourly System Marginal Price (SMP) (£/MWh)", yaxis_title="SMP (£/MWh)", color='green', line=False)

In [98]:
# Calculate the total cost for each hour based on the generators used and their outputs
cost_by_hour = [sum(gen.cost(output) for gen, output in gens_in_hour) for gens_in_hour in generators_by_hour]

title = f"Hourly Generation Cost (£)<br><sup>Total Day Cost (£{sum(cost_by_hour):,.2f})</sup>"
plot_hourly(cost_by_hour, title=title, yaxis_title="Cost (£)", color='orange')

In [99]:
start_up_cost_by_hour = [sum(gen.scost for gen in start_ups) for start_ups in start_up_by_hour]

title = f"Hourly Start-Up Cost (£)<br><sup>Total Day Start-Up Cost (£{sum(start_up_cost_by_hour):,.2f})</sup>"
plot_hourly(start_up_cost_by_hour, title=title, yaxis_title="Start-Up Cost (£)", color='magenta', line=False)

In [100]:
total_marginal_cost = sum(cost_by_hour)
total_start_up_cost = sum(start_up_cost_by_hour)
total_cost = total_marginal_cost + total_start_up_cost
print("TOTAL MARGINAL COST FOR THE DAY (GREEDY): £{:,.2f}".format(total_marginal_cost))
print("TOTAL START-UP COST FOR THE DAY (GREEDY): £{:,.2f}".format(total_start_up_cost))
print("TOTAL COST FOR THE DAY (GREEDY): £{:,.2f}".format(total_cost))

TOTAL MARGINAL COST FOR THE DAY (GREEDY): £135,500.00
TOTAL START-UP COST FOR THE DAY (GREEDY): £66,000.00
TOTAL COST FOR THE DAY (GREEDY): £201,500.00


### Part 2 — Unit Commitment with Linear Programming

Now implement the **full Unit Commitment problem** as a Mixed-Integer Linear Programme 
(MILP) using `PuLP`.

**Decision variables:**
- `u[g, t]` ∈ {0, 1} — whether generator `g` is online at hour `t`
- `p[g, t]` ≥ 0 — power output (MW) of generator `g` at hour `t`
- `v[g, t]` ∈ {0, 1} — whether generator `g` starts up at hour `t` (i.e., was off at t-1, on at t)

**Objective:** Minimise total cost:
$$\text{minimise} \sum_{g,t} \left( c_g \cdot p_{g,t} + SC_g \cdot v_{g,t} \right)$$

where $c_g$ is marginal cost and $SC_g$ is startup cost.

**Constraints:**
1. **Demand balance**: total generation must meet demand each hour
   $$\sum_g p_{g,t} = D_t \quad \forall t$$

2. **Capacity limits**: a generator can only produce between 0 and its capacity, and only 
   if it is online
   $$0 \leq p_{g,t} \leq P_g^{max} \cdot u_{g,t} \quad \forall g, t$$

3. **Startup detection**: a startup occurs when a unit goes from off to on
   $$v_{g,t} \geq u_{g,t} - u_{g,t-1} \quad \forall g, t > 0$$

4. **Minimum run time**: once started, a generator must run for at least `MRT_g` hours
   $$\sum_{\tau=t}^{\min(t + MRT_g - 1, T)} u_{g,\tau} \geq MRT_g \cdot v_{g,t} \quad \forall g, t$$

5. **Minimum stable generation**: a committed generator must produce at least its minimum stable output
$$p_{g,t} \geq P_g^{min} \cdot u_{g,t} \quad \forall g, t$$

**Output:**
- Optimal schedule: for each hour, which generators are committed and at what output
- Total optimal cost (including startup costs)
- Same stacked bar chart and SMP line plot as Part 1, now reflecting optimal dispatch
- The **commitment schedule** as a heatmap (generators × hours, shaded by output)

**Question to answer:** What is the total cost saving of the optimal UC solution versus 
the greedy merit order (accounting for startup costs in both)?

In [101]:
!pip install --quiet pulp
import pulp

In [135]:
def add_uclp_constraints(problem, generators, demand, u, v, p):
    T = len(demand)       # Number of hours (time periods)
    Ng = len(generators)  # Number of generators

    # Add the constraints for each generator at each hour
    for t in range(T):
        for i in range(Ng):
            gen = generators[i]

            # Capacity constraint: power output cannot exceed generator capacity when it's on
            problem += p[i, t] <= gen.capacity * u[i, t], f"Capacity_Generator_{i}_Hour_{t}"

            # Start-up detection: a start up occurs when a generator comes from off to on
            if t > 0:
                problem += v[i, t] >= u[i, t] - u[i, t-1], f"Start_Up_Detection_Generator_{i}_Hour_{t}"
            else:
                problem += v[i, t] >= u[i, t], f"Start_Up_Detection_Generator_{i}_Hour_{t}"  # For the first hour, any generator that is on is considered to have started up

            # Minimum runtime constraint: if a generator is started, it must run for at least its minimum runtime
            end_time = min(t + gen.min_runtime, T)  # Ensure we don't go beyond the time period
            problem += pulp.lpSum(u[i, k] for k in range(t, end_time)) >= gen.min_runtime * v[i, t], f"Minimum_Runtime_Generator_{i}_Hour_{t}"

            # Minimum output constraint: if a generator is on, it must produce at least its minimum output
            problem += p[i, t] >= gen.min_output * u[i, t], f"Minimum_Output_Generator_{i}_Hour_{t}"
            
        # Demand balance constraint: total generation must meet demand for each hour
        problem += pulp.lpSum(p[i, t] for i in range(Ng)) == demand[t], f"Demand_Balance_Hour_{t}"
    return problem


def solve_uclp(generators, demand):
    T = len(demand)       # Number of hours (time periods)
    Ng = len(generators)  # Number of generators

    # Initialise the optimization problem
    problem = pulp.LpProblem("UnitCommitment", pulp.LpMinimize)

    # Set up decision variables
    u = {(i, t): pulp.LpVariable(f"u_{i}_{t}", lowBound=0, upBound=1, cat='Binary') 
        for i in range(Ng) 
        for t in range(T)}                                # Binary variable, deciding whether a generator is on or off for each generator and hour
    v = {(i, t): pulp.LpVariable(f"v_{i}_{t}", lowBound=0, upBound=1, cat='Binary') 
        for i in range(Ng) 
        for t in range(T)}                                # Binary variable, deciding whether to start up a generator or not at each hour
    p = {(i, t): pulp.LpVariable(f"p_{i}_{t}", lowBound=0)# We could set an upper bound here based on the generator capacity, but we will rely on the BINDING capacity constraint to enforce that instead
        for i in range(Ng) 
        for t in range(T)}                                # Continuous variable, bounded below by 0 and above by max capacity, representing the power output of each generator at each hour


    # Objective function: minimize total cost (operating + start-up)
    problem += pulp.lpSum(
        generators[i].mcost * p[i, t] + generators[i].scost * v[i, t]
        for i in range(Ng) 
        for t in range(T)
    )

    # Add the constraints for each generator at each hour
    add_uclp_constraints(problem, generators, demand, u, v, p)

    # Solve the optimization problem
    problem.solve()

    return problem, p, u, v

problem, p, u, v = solve_uclp(generators, demand)
Ng, T = len(generators), len(demand)

Welcome to the CBC MILP Solver 
Version: 2.10.3 
Build Date: Dec 15 2019 

command line - /opt/homebrew/Caskroom/miniconda/base/envs/eda/lib/python3.10/site-packages/pulp/apis/../solverdir/cbc/osx/i64/cbc /var/folders/fq/ynhnpw_j05x2l8ypw293s7xw0000gp/T/41d3884f4cba44ba983a9e0482530004-pulp.mps -timeMode elapsed -branch -printingOptions all -solution /var/folders/fq/ynhnpw_j05x2l8ypw293s7xw0000gp/T/41d3884f4cba44ba983a9e0482530004-pulp.sol (default strategy 1)
At line 2 NAME          MODEL
At line 3 ROWS
At line 509 COLUMNS
At line 2606 RHS
At line 3111 BOUNDS
At line 3352 ENDATA
Problem MODEL has 504 rows, 360 columns and 1376 elements
Coin0008I MODEL read with 0 errors
Option for timeMode changed from cpu to elapsed
Continuous objective value is 189450 - 0.00 seconds
Cgl0003I 0 fixed, 0 tightened bounds, 33 strengthened rows, 10 substitutions
Cgl0003I 0 fixed, 0 tightened bounds, 3 strengthened rows, 0 substitutions
Cgl0004I processed model has 414 rows, 320 columns (224 integer (224

In [136]:
uclp_generators_by_hour = [[(generators[i], p[i, t].varValue) for i in range(Ng) if p[i, t].varValue > 0] for t in range(T)]

plot_stacked_generators(uclp_generators_by_hour, title="Hourly Generation by Generator Type (MW) - UCLP Solution", yaxis_title="Generation (MW)")

In [137]:
uclp_start_ups_by_hour = [[generators[i] for i in range(Ng) if v[i, t].varValue > 0] for t in range(T)]

plot_hourly([len(start_ups) for start_ups in uclp_start_ups_by_hour], title="Number of Start-Ups per Hour - UCLP Solution", yaxis_title="Number of Start-Ups", color='red', line=False)


# print which generators were started up in each hour
for t, start_ups in enumerate(uclp_start_ups_by_hour):
    if start_ups:  # Only print hours where there were start-ups
        gen_names = ", ".join(gen.name + f"({gen.type})" for gen in start_ups)
        print(f"Start ups at Hour {t}: {gen_names}")

Start ups at Hour 0: g1(nuclear)
Start ups at Hour 7: g2(coal)


In [138]:
uclp_start_ups_cost_by_hour = [sum([generators[i].scost for i in range(Ng) if v[i, t].varValue > 0]) for t in range(T)]
uclp_total_start_up_cost = sum(uclp_start_ups_cost_by_hour)
title = f"Hourly Start-Up Cost (£)<br><sup>Total Day Start-Up Cost (£{uclp_total_start_up_cost:,.2f})</sup>"
plot_hourly(uclp_start_ups_cost_by_hour, title=title, yaxis_title="Start-Up Cost (£)", color='magenta', line=False)

In [139]:
uclp_smps = [max(gen.mcost for gen, _ in gens_in_hour) if gens_in_hour else 0 for gens_in_hour in uclp_generators_by_hour]

plot_hourly(uclp_smps, title="Hourly System Marginal Price (SMP) (£/MWh) - UCLP Solution", yaxis_title="SMP (£/MWh)", color='green', line=False)    

In [140]:
uclp_total_cost = problem.objective.value()
uclp_total_marginal_cost = uclp_total_cost - uclp_total_start_up_cost
print("TOTAL MARGINAL COST FOR THE DAY (UCLP): £{:,.2f}".format(uclp_total_marginal_cost))
print("TOTAL START-UP COST FOR THE DAY (UCLP): £{:,.2f}".format(uclp_total_start_up_cost))
print("TOTAL COST FOR THE DAY (UCLP): £{:,.2f}".format(uclp_total_cost))
print("\n")
print("TOTAL SAVINGS WITH UCLP DISPATCH COMPARED TO GREEDY DISPATCH: £{:,.2f}".format(total_cost - uclp_total_cost))


TOTAL MARGINAL COST FOR THE DAY (UCLP): £137,100.00
TOTAL START-UP COST FOR THE DAY (UCLP): £58,000.00
TOTAL COST FOR THE DAY (UCLP): £195,100.00


TOTAL SAVINGS WITH UCLP DISPATCH COMPARED TO GREEDY DISPATCH: £6,400.00


In [142]:
schedule = {(generators[i].name, t): int(pulp.value(p[i, t])) for i in range(Ng) for t in range(T)}
df = pd.DataFrame([schedule[(gen.name, t)] for gen in generators] for t in range(T)).T
df.index = [gen.name + f" ({gen.type})" for gen in generators]
plot_schedule_heatmap(df)


### Part 3 — Analysis and Interpretation

1. **Price duration curve**: plot the System Marginal Price sorted from highest to lowest 
   for both the greedy and optimal solutions. What does the shape tell you?




In [143]:
series = {"UCLP": uclp_smps, "Greedy": smps}
plot_multiple_series(series, title="Hourly System Marginal Price (SMP) Comparison (£/MWh)", yaxis_title="SMP (£/MWh)", colors=['green', 'lightgreen'], line=True, bar=False)

2. **Startup cost impact**: run the MILP with startup costs set to zero. How does the 
   schedule change? What does this reveal about the role of commitment costs?

In [144]:
generators_no_start = [
    Generator('g1', 'nuclear', 400, 10, 0, 8, 150),
    Generator('g2', 'coal', 300, 25, 0, 4, 100),
    Generator('g3', 'gas-ccgt', 200, 45, 0, 2, 60),
    Generator('g4', 'gas-ocgt', 100, 80, 0, 1, 0),
    Generator('g5', 'oil', 50, 120, 0, 1, 0),
]

problem, pns, uns, vns = solve_uclp(generators_no_start, demand)

Welcome to the CBC MILP Solver 
Version: 2.10.3 
Build Date: Dec 15 2019 

command line - /opt/homebrew/Caskroom/miniconda/base/envs/eda/lib/python3.10/site-packages/pulp/apis/../solverdir/cbc/osx/i64/cbc /var/folders/fq/ynhnpw_j05x2l8ypw293s7xw0000gp/T/eb16bcdcad8847049216a3e4a05b8138-pulp.mps -timeMode elapsed -branch -printingOptions all -solution /var/folders/fq/ynhnpw_j05x2l8ypw293s7xw0000gp/T/eb16bcdcad8847049216a3e4a05b8138-pulp.sol (default strategy 1)
At line 2 NAME          MODEL
At line 3 ROWS
At line 509 COLUMNS
At line 2486 RHS
At line 2991 BOUNDS
At line 3232 ENDATA
Problem MODEL has 504 rows, 360 columns and 1376 elements
Coin0008I MODEL read with 0 errors
Option for timeMode changed from cpu to elapsed
Continuous objective value is 132000 - 0.00 seconds
Cgl0003I 0 fixed, 0 tightened bounds, 33 strengthened rows, 10 substitutions
Cgl0003I 0 fixed, 0 tightened bounds, 3 strengthened rows, 0 substitutions
Cgl0004I processed model has 274 rows, 226 columns (130 integer (130

In [145]:
uclp_total_cost_no_start = problem.objective.value()
print("TOTAL COST FOR THE DAY (UCLP, NO START-UP COSTS): £{:,.2f}".format(uclp_total_cost_no_start))

print("\nSCHEDULE (MW) FOR EACH GENERATOR AND HOUR (UCLP, NO START-UP COSTS):")
schedule_no_start = {(generators_no_start[i].name, t): int(pulp.value(pns[i, t])) for i in range(Ng) for t in range(T)}
df_ns = pd.DataFrame([schedule_no_start[(gen.name, t)] for gen in generators_no_start] for t in range(T)).T
df_ns.index = [gen.name + f" ({gen.type})" for gen in generators_no_start]
plot_schedule_heatmap(df_ns, title="Generator Power Output Schedule (MW) - UCLP Solution (No Start-Up Costs)")

print("SCHEDULE (MW) FOR EACH GENERATOR AND HOUR (UCLP, previous result):")
plot_schedule_heatmap(df)

TOTAL COST FOR THE DAY (UCLP, NO START-UP COSTS): £134,000.00

SCHEDULE (MW) FOR EACH GENERATOR AND HOUR (UCLP, NO START-UP COSTS):


SCHEDULE (MW) FOR EACH GENERATOR AND HOUR (UCLP, previous result):


Conclusion: start up costs give incentive for systems that are on to remain on.

3. **Scarcity hour**: identify the hour of peak demand. What is the SMP at that hour? 
   Is the system capacity-constrained? What would need to happen to demand or capacity 
   for the Oil peaker (£120/MWh) to set the price — and what would that do to the SMP?

In [146]:
import numpy as np
print("Hour of peak demand:", np.argmax(demand), "h, with demand of", demand[np.argmax(demand)], "MW")
print("Max system capacity (sum of generator capacities):", sum(gen.capacity for gen in generators), "MW")
print("Deficit to trigger oil generator (demand minus capacity of all other generators):", demand[np.argmax(demand)] - sum(gen.capacity for gen in generators if gen.name != 'g5'), "MW")
print("Headroom until full grid capacity is reached (capacity of all generators minus peak demand):", sum(gen.capacity for gen in generators) - demand[np.argmax(demand)], "MW")
print("SMP if oil generator is triggered on (marginal cost of oil generator):", next(gen.mcost for gen in generators if gen.name == 'g5'), "£/MWh")

Hour of peak demand: 19 h, with demand of 625 MW
Max system capacity (sum of generator capacities): 1050 MW
Deficit to trigger oil generator (demand minus capacity of all other generators): -375 MW
Headroom until full grid capacity is reached (capacity of all generators minus peak demand): 425 MW
SMP if oil generator is triggered on (marginal cost of oil generator): 120 £/MWh


### Part 4 — Lagrange Multipliers, Dual Variables and Shadow Prices

In unconstrained optimisation you minimise a function $f(x)$ freely. In constrained optimisation you minimise $f(x)$ subject to $g(x) = b$ — and Lagrange multipliers are the tool that bridges the two.

The core idea: attach a multiplier $\lambda$ to each constraint and fold it into the objective, forming the **Lagrangian**:

$$\mathcal{L}(x, \lambda) = f(x) - \lambda \cdot (g(x) - b)$$

At the optimum, $\lambda$ tells you exactly how sensitive the optimal objective value is to a marginal relaxation of the constraint:

$$\lambda = \frac{\partial f^*}{\partial b}$$

In plain terms: **if you loosen constraint $g$ by one unit, the optimal objective changes by $\lambda$**. This is why $\lambda$ in this power market example is called a **shadow price**: it is the rate at which the optimal cost changes if the constraint's bound is relaxed by one unit — i.e., the implicit price the system places on that resource.

**Why this matters in power markets**

Every constraint in the UC problem has a shadow price with a direct economic interpretation:

| Constraint | Shadow Price Meaning |
|---|---|
| Demand balance at hour $t$ | True marginal cost of serving 1 extra MW at hour $t$ — the true SMP, which may differ from the naive SMP when commitment decisions leave cheaper generators with spare capacity. |
| Capacity of generator $g$ at hour $t$ | Cost decrease if generator $g$ had 1 MW more capacity at hour $t$ — i.e. the **capacity price** |
| Minimum stable generation | Cost savings from lowering the operating floor by 1 MW |
| Minimum run time | Cost savings from freeing a generator to switch off 1 hour earlier |


**The MILP complication**

Shadow prices are well-defined for continuous linear programmes (LPs). The UC problem is a MILP — its binary variables make it non-convex, and strict duality theory does not apply directly. The standard practical approach is to **fix the binary variables** at their optimal values from the MILP solution and re-solve the resulting LP. This LP has the same optimal dispatch but now admits well-defined dual variables, which are meaningful as local price signals around the optimal solution.

---

#### Tasks

**4.1 — Extract shadow prices from the LP relaxation**

Take the optimal binary solution $u^*$ and $v^*$ from Part 2. Fix these values and re-solve as a pure LP by replacing all binary variables with fixed constants. Extract the dual values representing shadow prices.

**Question**: *How do these shadow prices compare to the SMP values you computed in Part 2? Are they identical? If not, why might they differ?*

In [159]:
fixed_problem = pulp.LpProblem("UC_LP_Relaxation", pulp.LpMinimize)

# Re-declare p as the only free (continuous) variable
p = {(i, t): pulp.LpVariable(f"p_{i}_{t}", lowBound=0)#, upBound=generators[i].capacity) 
     for i in range(Ng) 
     for t in range(T)}          
                      
# u and v fixed at their MILP-optimal values
u_optimal = {(i, t): int(pulp.value(u[i, t])) for i in range(Ng) for t in range(T)}
v_optimal = {(i, t): int(pulp.value(v[i, t])) for i in range(Ng) for t in range(T)}

# Objective function: minimize total cost (operating + start-up)
fixed_problem += pulp.lpSum(
    generators[i].mcost * p[i, t] + generators[i].scost * v_optimal[i, t]
    for i in range(Ng) 
    for t in range(T)
)

# Add the constraints for each generator at each hour, using the fixed u and v values
add_uclp_constraints(fixed_problem, generators, demand, u_optimal, v_optimal, p)

# Solve the fixed problem
fixed_problem.solve()

# Extract dual variables via constraint.pi
dual_variables = {}
for c in fixed_problem.constraints.values():
    dual_variables[c.name] = c.pi


Welcome to the CBC MILP Solver 
Version: 2.10.3 
Build Date: Dec 15 2019 

command line - /opt/homebrew/Caskroom/miniconda/base/envs/eda/lib/python3.10/site-packages/pulp/apis/../solverdir/cbc/osx/i64/cbc /var/folders/fq/ynhnpw_j05x2l8ypw293s7xw0000gp/T/c591c3562b194e518d11258700cabdae-pulp.mps -timeMode elapsed -branch -printingOptions all -solution /var/folders/fq/ynhnpw_j05x2l8ypw293s7xw0000gp/T/c591c3562b194e518d11258700cabdae-pulp.sol (default strategy 1)
At line 2 NAME          MODEL
At line 3 ROWS
At line 389 COLUMNS
At line 870 RHS
At line 1255 BOUNDS
At line 1256 ENDATA
Problem MODEL has 384 rows, 120 columns and 360 elements
Coin0008I MODEL read with 0 errors
Option for timeMode changed from cpu to elapsed
Presolve 14 (-370) rows, 28 (-92) columns and 28 (-332) elements
Perturbing problem by 0.001% of 25 - largest nonzero change 3.4176054e-06 ( 2.0746445e-05%) - largest zero change 0
0  Obj 116711.51 Primal inf 2037.2001 (14)
14  Obj 137100.01
Optimal - objective value 137100

In [148]:
# Print the dual variables (shadow prices) for each constraint
print("Dual variables (shadow prices) for each constraint:")
for name, value in dual_variables.items():
    if value != 0:  # Only print non-zero dual variables for clarity
        print(f"{name}: {value:.2f} £/MWh")


# Plot smts and dual variables for demand balance constraints
demand_balance_duals = [dual_variables.get(f"Demand_Balance_Hour_{t}", 0) for t in range(T)]
series = {"Naive SMP": uclp_smps, "Dual Variable (Demand Balance)": demand_balance_duals}
plot_multiple_series(series, title="Hourly System Marginal Price (SMP) vs Dual Variable for Demand Balance Constraint (£/MWh)", yaxis_title="£/MWh", colors=['green', 'blue'], line=True, bar=False)

Dual variables (shadow prices) for each constraint:
Demand_Balance_Hour_0: 10.00 £/MWh
Demand_Balance_Hour_1: 10.00 £/MWh
Demand_Balance_Hour_2: 10.00 £/MWh
Demand_Balance_Hour_3: 10.00 £/MWh
Demand_Balance_Hour_4: 10.00 £/MWh
Demand_Balance_Hour_5: 10.00 £/MWh
Demand_Balance_Hour_6: 10.00 £/MWh
Minimum_Output_Generator_1_Hour_7: 15.00 £/MWh
Demand_Balance_Hour_7: 10.00 £/MWh
Minimum_Output_Generator_1_Hour_8: 15.00 £/MWh
Demand_Balance_Hour_8: 10.00 £/MWh
Capacity_Generator_0_Hour_9: -15.00 £/MWh
Demand_Balance_Hour_9: 25.00 £/MWh
Capacity_Generator_0_Hour_10: -15.00 £/MWh
Demand_Balance_Hour_10: 25.00 £/MWh
Capacity_Generator_0_Hour_11: -15.00 £/MWh
Demand_Balance_Hour_11: 25.00 £/MWh
Capacity_Generator_0_Hour_12: -15.00 £/MWh
Demand_Balance_Hour_12: 25.00 £/MWh
Capacity_Generator_0_Hour_13: -15.00 £/MWh
Demand_Balance_Hour_13: 25.00 £/MWh
Minimum_Output_Generator_1_Hour_14: 15.00 £/MWh
Demand_Balance_Hour_14: 10.00 £/MWh
Minimum_Output_Generator_1_Hour_15: 15.00 £/MWh
Demand_Balance


This is a discussion over the naive SMP and the true SMP. When nuclear is at full capacity (9h-13h and 16h-21h) or when coal is not committed (0-6h and 22-23h), the true SMP matches the naive SMP.

However, when nuclear is not a full capacity but coal is committed at minimum stable generation (7-8h), the cost of adding another MW to generation is that of nuclear. The dual variables better this behaviour compared to a naive SMP computation.


**4.2 — Capacity shadow prices**

Extract the dual variable on the capacity constraint for each generator at each hour:

```python
for i in range(Ng):
    for t in range(T):
        mu = lp_problem.constraints[f"Capacity_Generator_{i}_Hour_{t}"].pi
```

Plot a heatmap of capacity shadow prices (generators × hours).

**Questions to answer:**
- Which generators have non-zero capacity shadow prices, and at which hours?

- What are the two conditions required for a capacity shadow price to be non-zero — and why is binding alone not sufficient?

- At the hours of highest demand, which generator's capacity is most valuable? What would it be worth to add 10 MW (assuming linearity) to that generator?

In [162]:
capacity_duals = {(i, t): dual_variables.get(f"Capacity_Generator_{i}_Hour_{t}", 0) for i in range(Ng) for t in range(T)}

df = pd.DataFrame(
    [[capacity_duals.get((i, t), 0) for t in range(T)] for i in range(Ng)],
    index=[gen.name + f" ({gen.type})" for gen in generators]
)

plot_schedule_heatmap(df, title="Generator Capacity Shadow Prices (£/MW) - UCLP Solution", zmin=-15, zmax=15, colors="RdYlGn_r")

- Gas and oil (g3-5) never need to be committed for this specific demand and system, so will have trivial capacity shadow price.

- The combined capacity of coal and nuclear is 700W. Since demand at any hour never exceeds this, adding 1 unit of capacity to coal has no effect on the generator's output and therefore has no effect on the total cost. It is, therefore, trivial everywhere.

- At hours where Coal is committed but pinned at its minimum stable generation (100 MW), an extra MW from Nuclear can't be used **because we can't push Coal below 100 while it's on**, so the substitution is blocked. At those specific hours, the capacity shadow price really is zero. But at **any hour where Coal is producing above 100 MW,** the extra MW of Nuclear capacity would displace expensive Coal, and the shadow price is non-zero. More specifically, one extra MW of Nuclear capacity displaces one MW of Coal, saving £25 − £10 = £15/MWh. That's the expected capacity shadow price.

*Question: What are the two conditions required for a capacity shadow price to be non-zero — and why is binding alone not sufficient?*

A constraint is binding when the optimal solution sits exactly on the contraint's boundary and the inequality holds as an equality.
A non-zero capacity shadow price requires two conditions: 
1) the capacity constraint is binding — the generator is at full output, and 
2) a more expensive committed generator is producing above its minimum stable generation, so that an extra MW of cheap capacity can displace an expensive one.

Binding alone is not sufficient because if no substitution is available either because no expensive generator is running, or because it's pinned at its operating floor (the latter would occur at a demand of exactly 400MW + 100MW = 500MW). In these cases the extra capacity has nowhere to go and the cost doesn't change.


*Question: At the hours of highest demand, which generator's capacity is most valuable? What would it be worth to add 10 MW (assuming linearity) to that generator?*

At peak hours (16h-21h), the capacity shadow price of nuclear is -£15/MWh, meaning that adding 10MW of capacity to nuclear would allow for £150 At peak hours (16h–21h), the capacity shadow price of Nuclear is £15/MWh, meaning that each additional MW of Nuclear capacity would displace £15/h of more expensive Coal generation. Adding 10 MW (assuming linearity) would save £15/MWh * 10MW = £150/h at each of those hours. All other generators have zero capacity shadow prices, making Nuclear's capacity the most valuable.

**4.3 — Interpreting the minimum stable generation constraint**

Extract the dual variable on the minimum stable generation constraint for Coal (G2) during the midday dip hours (14-15h):

**Questions to answer:**
- Is this constraint binding during the dip? What does that tell you about Coal's dispatch at those hours?

- What does the shadow price tell you about the cost of the minimum stable generation requirement? In other words, how much would the system save if Coal could operate at zero output rather than 100 MW while remaining committed?

- Connect this back to the UC saving from Part 2: the optimal solution kept Coal on through the dip at minimum stable output. The shadow price here quantifies exactly what that decision cost per MW of floor output.

In [165]:
minimum_stable_duals = {(i, t): dual_variables.get(f"Minimum_Output_Generator_{i}_Hour_{t}", 0) for i in range(Ng) for t in range(T)}


df = pd.DataFrame(
    [[minimum_stable_duals.get((i, t), 0) for t in range(T)] for i in range(Ng)],
    index=[gen.name + f" ({gen.type})" for gen in generators]
)

plot_schedule_heatmap(df, title="Generator Minimum Output Shadow Prices (£/MW) - UCLP Solution", zmin=-15, zmax=15, colors="RdYlGn_r")

print("Minimum output shadow prices of g2 at 14h:", minimum_stable_duals.get((1, 14), 0), "£/MW")
print("Minimum output shadow prices of g2 at 15h:", minimum_stable_duals.get((1, 15), 0), "£/MW")

Minimum output shadow prices of g2 at 14h: 15.0 £/MW
Minimum output shadow prices of g2 at 15h: 15.0 £/MW


*Question: Is this constraint binding during the dip? What does that tell you about Coal's dispatch at those hours?*
Since the shadow price is non-trivial, the minimum output generation of coal at the dip hours must be binding (a necessary condition.) 

x

*Question: What does the shadow price tell you about the cost of the minimum stable generation requirement? In other words, how much would the system save if Coal could operate at zero output rather than 100 MW while remaining committed?*

The minimum stable generation shadow price indicates how much the total cost would decrease per MW reduction in a generator's output floor. At the dip hours, Coal is pinned at its 100 MW floor while Nuclear has spare capacity (~280 MW against a 400 MW cap). Each MW reduction in Coal's floor allows Nuclear to produce one more MW at £10 instead of Coal at £25, saving £15/MWh (matching the heatmap above). Reducing the floor from 100 MW to zero would save £1,500/h, or £3,000 across the two dip hours. This relies on Nuclear having sufficient headroom to absorb the substitution; if Nuclear were also at capacity, the saving would be smaller or zero.

x

*Question: Connect this back to the UC saving from Part 2 where the optimal solution kept Coal on through the dip at minimum stable output. The shadow price here quantifies exactly what that decision cost per MW of floor output.*

Keeping Coal committed during the dip hours costs an extra £3,000 in above-optimal operating cost (£1,500/h × 2 hours). However, the UC algorithm accepts this because the alternative of shutting Coal down and restarting it later would incur an £8,000 startup cost, a net loss of £5,000. The shadow price quantifies exactly what the "keep it on" decision costs per MW of floor output.



## Conclusions

- The 425 MW of headroom confirms the system is not capacity-constrained at peak.

- At peak hour (19h), the SMP is £25/MWh, set by Coal as the most expensive generator dispatched.

- Fluctuating demand creates little incentive to keep generators running at high output throughout the day. This is heavily influenced by grid constraints such as startup costs and minimum runtime, which the UC formulation captures directly.

- Naturally, I would suspect a similar trend would emerge at the annual scale, particularly in countries with seasonal weather variation, where heating and cooling demand introduces a predictable but significant demand cycle on top of the daily one.

- For Oil to set the price, demand at peak would need to rise by 375 MW. The SMP would then jump to £120/MWh, nearly 5× the current peak price.

- This illustrates why scarcity events carry such financial weight, and why grid stress assessment matters since a relatively small demand increase can trigger a discontinuous and dramatic price spike.

- The demand balance shadow prices give us the System Marginal Price directly from duality theory, rather than reading off the marginal cost of the last dispatched generator. This turns out to be more robust because it accounts for the full constraint structure (min stable generation, min run times) that the naive merit-order reading misses

- Capacity shadow prices reveal which generators' limits are actively costing the system money. Here, Nuclear is the only one with non-zero values (£15/MWh at hours where Coal can be displaced), confirming it as the most valuable unit to expand. 

- The minimum stable generation shadow prices make the UC algorithm's commitment tradeoffs explicit. Keeping Coal on during the midday dip costs £3,000 in avoidable operating cost, but shutting it down and restarting it would cost £8,000, so the algorithm is making the right call. 

- The last conclusion is particularly satisfying because instead of just trusting the solver's output, shadow prices let us put a price tag on each constraint and see exactly why the algorithm made the decisions it did.

## Personal Reflections


- This exercise introduced me to the basic setup of a power grid optimisation problem, and it was extremely useful for getting acquainted with a domain I hadn't encountered before.

- It was a nice refresher on constrained optimisation, and offered an intuitive perspective on why power markets are modelled the way they are.

- It was also helpful to study Lagrange multipliers and see how they can be used to justify and quantify the market decisions made by the optimisation.

- I was also introduced to PuLP, which makes formulating and solving MILPs far more straightforward than building them from scratch.

- However, it would still be interesting to implement the MILP from scratch as a more rigorous refresher on the underlying mechanics.

- MILP is not known for its scalability, so a comparative study against other solvers or formulations would be a natural follow-up.

- That said, this problem is not complex enough to expose those limitations. A more meaningful next step would be to explore advanced power market examples. Potentially with real collected data from Kaggle or HuggingFace. Nonetheless, an enjoyable little exercise.